
### 📅 RAG 성능 최적화: 4주 실험 로드맵

<br>

#### **1주차: 데이터의 질(Data Quality) 개선 실험**

가장 먼저 해결해야 할 것은 `similarity_k10`에서 발견된 **Field Coverage** 문제입니다. 모델이 아무리 똑똑해도 읽는 데이터가 부실하면 한계가 있습니다.

##### 1. 노이즈 제거 및 텍스트 정규화 (Cleaning)

청크 내에 불필요한 특수문자가 많으면 임베딩 모델이 의미를 파악하는 데 방해가 됩니다.

* **불필요한 공백/줄바꿈 정리**: 연속된 공백이나 의미 없는 줄바꿈(`\n\n\n`)을 단일 공백으로 치환합니다.
* **특수 기호 정제**: 목차를 나타내는 점(`...`)이나 페이지 번호, 헤더/푸터 정보를 제거합니다.
* **인코딩 확인**: 한글 깨짐 현상을 방지하기 위해 UTF-8 인코딩을 준수합니다.

##### 2. 메타데이터 기반 컨텍스트 삽입 (Context Injection)

청크가 작아질수록(300자 등) 해당 문장의 주어가 무엇인지 알기 어려워집니다. 전처리 단계에서 이를 보강합니다.

* **제목 결합**: 각 청크의 맨 앞에 `[문서 제목] + [섹션명]`을 강제로 삽입합니다.
* *예: "[한영대학교 특성화 사업] 사업 목적: ..."*


* **Chunk 구분자 활용**: `RecursiveCharacterTextSplitter` 사용 시 `["\n\n", "\n", " ", ""]`와 같이 구분자 우선순위를 명확히 주어 의미 단위가 잘리지 않게 합니다.

##### 3. 표(Table) 및 구조화된 데이터 처리

로드맵에 언급된 **'Field Coverage'** 문제를 해결하기 위해 가장 중요한 부분입니다.

* **Markdown 변환**: 표 데이터를 단순 텍스트로 읽으면 행/열 관계가 깨집니다. 전처리 시 표를 **Markdown 형식**으로 변환하면 LLM이 구조를 훨씬 잘 파악합니다.

* **Summary 생성**: 표나 복잡한 수치 데이터는 별도의 '요약문'을 생성하여 `chunk_type: summary`로 저장해 둡니다.

##### 4. QA 데이터셋(Evaluation Set) 정제

실험의 기준이 되는 `qa.parquet`의 질도 중요합니다.

* **Ground Truth 검증**: `retrieval_gt`의 `doc_id`가 `corpus.parquet`에 실제로 존재하는지, 그리고 해당 질문을 풀기에 충분한 정보를 담고 있는지 전수 검사합니다.

* **질문 다양화**: 단순 사실 확인형 질문 외에 "사업의 기대 효과를 요약해줘"와 같은 요약형 질문을 포함하여 `summary` 청크의 효용성을 테스트합니다.


<br>

#### **2주차: 검색 알고리즘(Retrieval) 고도화**

단순 유사도 검색을 넘어, 질문의 의도를 더 잘 파악하고 방대한 문서군에서 정답이 포함된 문서를 정확히 집어내는 기술을 도입합니다.

##### **1. 베이스라인 구축 및 파라미터 최적화**

* **Naive Retrieval 설정**: 고도화 전, 가장 기본적인 형태의 유사도 검색을 구현하여 성능 지표(Hit@k, MRR)의 기준점을 잡습니다.

* **Top-k 실험**: LLM에 전달할 컨텍스트의 양을 결정하는 `k` 값을 3, 5, 10으로 변화시키며 답변의 정확도와 토큰 비용 사이의 트레이드오프를 관찰합니다.

* **MMR(Maximum Marginal Relevance)**: 유사도만 따지는 것이 아니라, 검색 결과 간의 중복을 피하고 '다양성'을 확보하여 LLM이 더 폭넓은 정보를 참고하게 합니다.

##### **2. Query Transformation (질문 최적화)**

* **Query Rewriting**: "그 사업 언제까지야?"와 같은 모호한 질문을 LLM을 통해 "한영대학교 특성화 사업의 최종 사업 종료일은 언제인가?"와 같이 검색에 최적화된 문장으로 재작성합니다.

* **Multi-Query**: 하나의 질문을 서로 다른 관점의 3~5개 질문으로 확장하여 검색 누락(False Negative)을 최소화합니다.

* **HyDE(Hypothetical Document Embeddings)**: 질문에 대해 LLM이 가상의 답변을 먼저 생성하게 하고, 그 '가상 답변'과 유사한 실제 문서를 검색하여 의미적 간극을 메웁니다.

##### **3. 메타데이터 필터링 및 하이브리드 검색**

* **지능형 필터링**: 여러 문서가 섞여 있을 때, 질문 속 키워드(예: '한영대', '24년도')를 추출하여 관련 문서만 검색 범위로 한정합니다. 이때 사용자의 오타나 불완전한 명칭을 고려한 유연한 필터링 로직을 실험합니다.

* **Hybrid Search (Dense + Sparse)**:

    * **Dense**: 의미적 유사성 파악 (Vector DB)

    * **Sparse**: 고유 명사나 핵심 키워드 일치 확인 (BM25)

    * 두 결과를 RRF(Reciprocal Rank Fusion)를 통해 결합하여 검색 품질을 극대화합니다.

##### **4. Re-ranking (재정렬) 도입**

* **Cross-Encoder 활용**: Vector DB가 1차로 뽑아온 상위 후보(예: k=20)를 **BGE-Reranker**와 같은 고성능 모델로 다시 평가합니다.

* **성능 검증**: 단순 검색 대비 `nDCG` 및 `Hit Rate` 지표가 얼마나 상승하는지 측정하고, 재정렬 단계에서 발생하는 지연 시간(Latency)이 사용자 경험에 미치는 영향을 분석합니다.

<br>

#### **3주차: 답변 생성(Generation) 및 환각 제어**

`grounded_token_ratio`(근거 기반 토큰 비율)를 높여 답변의 신뢰도를 확보하고, 시스템의 효율성을 극대화하는 단계입니다.



##### **1. Generation 기능 구현 및 모델 최적화**

LLM을 선정할 때는 **텍스트 생성 능력** 과 **추론 비용** 사이의 균형점을 찾는 것이 핵심입니다.

* **모델 벤치마크 및 시나리오 비교**:
    * `gpt-5-mini`, `gpt-5-nano`와 같은 경량형 모델부터 `gemma-4` 등 로컬 모델까지 `avg_llm_faithfulness`(평균 충실도) 수치를 비교하여 프로젝트 목적에 맞는 최적의 모델을 선정합니다.


* **디코딩 파라미터 설정**:
    * 답변의 창의성보다 정확도가 중요한 RAG 시스템의 특성을 고려하여 `temperature`와 `top_p`를 낮게 설정(예: 0.1~0.3)합니다.
    * `max_tokens`를 통해 불필요한 장황함을 방지하고 비용을 관리합니다.


* **토큰 사용량 최적화**:
    * 성능을 유지하면서도 입력 토큰을 줄일 수 있는 프롬프트 압축 기법을 적용하여 운영 비용을 절감합니다.



#####  **2. 신뢰도 높은 답변을 위한 프롬프트 엔지니어링**

Retrieval 과정에서 확보한 컨텍스트를 모델이 얼마나 충실히 반영하느냐가 시스템의 성패를 결정합니다.

* **최적의 프롬프트 구성**:
    * "주어진 컨텍스트 내에서만 답변하고, 모르는 내용은 '모른다'고 답하라"는 제약 조건을 명시하여 환각(Hallucination)을 제어합니다.


* **Chain of Thought (CoT)**:
    * "단계별로 사고하여 답변을 도출하라"는 지침을 추가하여 복잡한 논리 구조에서도 답변의 일관성을 유지합니다.


* **Self-Correction (자기 수정)**:
    * 모델이 생성한 답변이 근거 문서와 일치하는지 스스로 검토하게 하여 `groundedness`를 자발적으로 높이도록 설계합니다.


* **톤 및 스타일 조정**:
    * 사용자 페르소나에 맞춰 응답의 톤(전문적/친절함)과 서술 방식(개조식/줄글)을 커스텀합니다.


##### **3. 대화 맥락(Context) 유지 및 관리**

단발성 질의응답을 넘어 사용자와의 지속적인 상호작용을 위해 대화 흐름을 관리합니다.

* **Conversation Memory**:
    * 이전 대화의 핵심 요약본을 컨텍스트에 포함시켜, 대화가 길어져도 맥락을 잃지 않고 자연스러운 질의응답이 가능하도록 구현합니다.


* **Query Re-writing**:
    * 사용자의 모호한 질문을 대화 이력에 기반해 명확한 질문으로 재구성하여 검색 성능을 보완합니다.

<br>

#### **4주차: 릴리즈 게이트 통과 및 최종 최적화**

실제 서비스 수준의 안정성을 확보하고, 정교한 성능 지표를 통해 베스트 설정을 확정하는 단계입니다.


##### **1. 다각도 성능 평가 및 지표 수립**

단순한 수치를 넘어, RAG 시스템이 실제 비즈니스 요구사항을 얼마나 충실히 이행하는지 평가합니다.

* **평가 기준 (Evaluation Metrics)**:

    * **추출 정밀도(Extraction Accuracy)**: 단일 문서에서 핵심 정보를 정확히 추출하는지 평가합니다.

    * **종합 능력(Synthesis)**: 여러 문서에 흩어진 정보를 논리적으로 병합하여 답변하는지 확인합니다.
    
    * **맥락 유지(Contextual Awareness)**: 후속 질문(Follow-up)의 의도를 파악하고 대화 흐름을 이어가는지 검증합니다.
    
    * **환각 방지(Grounding & Refusal)**: 문서에 없는 내용에 대해 "모른다"고 명확히 답하는지 체크합니다.


* **평가 방법론**: 위 기준들을 `Faithfulness`, `Answer Relevance`, `Context Precision` 등의 정량적 지표(RAGAS 등 활용)로 수치화하여 관리합니다.


##### **2. 다양한 질문 세트를 활용한 스트레스 테스트**

도메인 특화 데이터(예: 공공기관 제안요청서)를 기반으로 실제 사용자가 던질 법한 복잡한 질문 세트를 구성해 성능을 검증합니다.

* **질문 시나리오 예시**:

    * **단일 정보 추출**: "국민연금공단의 이러닝 시스템 사업 요구사항 정리해 줘."
    * **상세 심층 질문**: "콘텐츠 개발 관리 요구사항에 대해 더 자세히 알려 줘."
    * **비교 분석**: "고려대 차세대 포털과 GIST 학사 시스템 사업의 응답 시간 요구사항을 비교해 줘."
    * **부재 정보 확인**: "기초과학연구원 사업에서 'AI 기반 예측' 요구사항이 문서에 명시되어 있는가?"


##### **3. Release Gate 임계값(Threshold) 도전**

안정적인 서비스 배포를 위해 통과 기준을 엄격하게 상향 조정합니다.

* **지표 상향**: 현재의 통과 기준(예: `Hit@5` 0.83)을 0.9 이상으로 높이고, 이를 달성하기 위해 `Top-k` 파라미터나 리랭킹(Re-ranking) 알고리즘을 튜닝합니다.

* **Gold Standard 구축**: 검증된 정답 셋(Ground Truth)을 바탕으로 배포 전 최종 전수 조사를 실시합니다.


##### **4. 비용 및 속도 최적화 (Efficiency)**

품질만큼 중요한 것이 운영 효율성입니다. 사용자 경험과 비용을 동시에 잡습니다.

* **Prompt Compression (프롬프트 압축)**: `avg_total_tokens`를 줄이면서도 정보 손실을 최소화하는 기술을 실험하여 추론 비용을 절감합니다.

* **Latency 제어 (P95 최적화)**:
    * 전체 응답의 95%가 특정 시간 안에 도달하도록 지연 시간을 관리합니다.
    * 사용자가 체감하는 대기 시간을 줄이기 위해 **스트리밍(Streaming)** 출력 방식을 최적화하고, 필요 시 비동기 처리를 도입합니다.

* **성능과 속도의 트레이드오프(Trade-off) 관리**: 응답 속도가 일정 수준 이하로 떨어지지 않도록 모델 사이즈와 임베딩 복잡도를 재조정합니다.

<br>

##### 💡 실험 팁

1. **지표 중심 사고 (Metric-Driven)**: "답변이 더 자연스러워졌어요"라는 주관적 평가 대신, "nDCG가 0.05 상승했고 Groundedness가 10% 개선되었습니다"라고 말할 수 있어야 합니다.

2. **실패한 질문(Error Analysis) 집중**: 평가 결과에서 `FAIL`이 난 질문(예: 아까 본 q3 후속질문)들만 모아서 왜 실패했는지 역추적하는 과정이 실력을 가장 많이 키워줍니다.

3. **Gate의 활용**: 실험 중간중간 `check_release_gate.py`를 실행하며, 본인의 실험 결과가 "배포 가능한 수준"인지 수시로 체크하게 하세요.


---

#### (참고) 🛠️ RAG 6단계 검색 메커니즘 설명

단순한 검색을 넘어, 대규모 데이터셋에서 정답을 날카롭게 추려내는 **하이브리드 검색 시스템** 의 전형적인 모습입니다. 각 단계가 왜 필요한지, 어떤 역할을 하는지 핵심 위주로 설명해 드릴게요.


##### ① 메타데이터 필터 (Metadata Filter) 🧹

* **역할**: 검색 범위를 확 줄여주는 '사전 거름망'입니다.

* **설명**: 8,000개에 가까운 전체 문서에서 유사도를 측정하면 속도가 느리고 오답이 섞일 수 있습니다. 따라서 질문과 관련된 특정 기관(예: 한국가스공사)이나 연도(예: 2024년)만 SQL의 `WHERE` 절처럼 딱딱하게(Hard Filter) 먼저 골라냅니다.

* **비유**: 도서관 전체에서 책을 찾기 전, 해당 '카테고리 서가'로 이동하는 단계입니다.

##### ② Dense 검색 (의미 매칭) 🧠

* **역할**: 문장의 '의미'와 '맥락'이 비슷한 것을 찾습니다.

* **설명**: `bge-m3` 같은 임베딩 모델을 사용합니다. "점심 메뉴 추천"이라고 물었을 때 "식사", "음식", "식당"처럼 단어는 달라도 뜻이 통하는 문서를 벡터 유사도로 찾아냅니다.

##### ③ Sparse 검색 (키워드 매칭) 🔍

* **역할**: '고유명사'나 '정확한 단어'를 잡아냅니다.

* **설명**: 전통적인 BM25 알고리즘을 씁니다. Dense 검색이 놓치기 쉬운 품번, 사람 이름, 특정 규정 번호 등 토씨 하나 안 틀리고 똑같은 키워드가 들어간 문서를 찾아내어 의미 검색의 약점을 보완합니다.

##### ④ RRF 융합 (Rank Fusion) 🤝

* **역할**: Dense와 Sparse의 **검색 결과를 공정하게 합칩니다.**

* **설명**: 의미 검색 결과 15개와 키워드 검색 결과 15개를 합쳐서 순위를 다시 매깁니다. 양쪽 검색 결과에서 공통적으로 상위에 오른 문서가 최종 상위권으로 올라가게 됩니다. (중복 제거 포함)

##### ⑤ Soft Boost (가점 부여) 🚀

* **역할**: 특정 조건에 맞는 문서에 '보너스 점수'를 줍니다.

* **설명**: 단순히 텍스트만 보는 게 아니라, 표(Table)가 포함된 문서나 특정 섹션, 혹은 숫자가 많이 포함된 데이터에 가중치(최대 0.15)를 주어 LLM이 답변하기 좋은 '풍부한 정보'를 위로 올립니다.

##### ⑥ Top-K 최종 전달 (Context Injection) 🎯

* **역할**: LLM이 읽기 가장 좋은 **최적의 양** 만 전달합니다.

* **설명**: 수천 개에서 시작한 후보를 가장 점수가 높은 **최종 5개** 로 압축합니다. LLM에게 "여기 [1]번부터 [5]번까지의 정보가 있으니 이걸 보고 답변해"라고 전달하며 프로세스가 마무리됩니다.

<br>

##### 💡 각 단계별 요약

이 메커니즘에서 가장 중요한 점은 '필터링 → 검색 → 결합 → 최적화'의 단계를 거치며 노이즈를 제거한다는 것입니다.

* **1단계**에서 수천 개를 수백 개로 줄이고,

* **2~3단계**에서 후보를 30개(15+15)로 압축한 뒤,

* **4~6단계**에서 정예 멤버 5개만 선발하는 구조입니다.
